# LEGO Sorter Pipeline - Interactive Controller

This notebook provides an interactive interface to control the Blender-based LEGO sorting simulation via MCP.

**Architecture**: Python Notebook → MCP Client → Blender (localhost:9876)

**Benefits**:
- Run individual pipeline steps interactively
- Add documentation and visualizations inline
- Experiment with different configurations
- Keep Blender scripts as separate .py files (version control friendly)

## Prerequisites

1. Blender must be running
2. BlenderMCP addon enabled (Press N → BlenderMCP tab → "Connect to Claude")
3. MCP server listening on localhost:9876

## Setup and Connection Test

In [15]:
# Import required modules
import sys
import os

# Add utils to path
sys.path.insert(0, os.path.join(os.getcwd(), "utils"))

from utils.blender_mcp_client import BlenderMCPClient

# Initialize client
client = BlenderMCPClient(timeout=300)

# Test connection
if client.test_connection():
    print("✅ Successfully connected to Blender MCP server")
    print(f"   Host: {client.host}:{client.port}")
    print(f"   Timeout: {client.timeout}s")
else:
    print("❌ Failed to connect to Blender MCP server")
    print("   Make sure Blender is running with BlenderMCP addon enabled")

✅ Blender MCP server is running on localhost:9876
✅ Successfully connected to Blender MCP server
   Host: localhost:9876
   Timeout: 300s


## Pipeline Step 1: Clear Scene

Reset the Blender scene by removing all objects and cleaning up collections.

**Script**: `blender/clear_scene.py`  
**Purpose**: Start with a clean slate before building the simulation

In [16]:
# Clear the Blender scene
success = client.execute_script_file(
    'blender/clear_scene.py',
    'Clear Scene',
    timeout=60
)

if success:
    print("\n✅ Scene cleared successfully")
else:
    print("\n❌ Failed to clear scene")

📝 Executing script: blender/clear_scene.py
✅ Successfully executed Clear Scene!
Result:
🧹 Clearing Blender scene...
✅ Removed 3 objects from scene
✅ Cleaned up 2 empty collections
✅ Scene cleared successfully


✅ Scene cleared successfully


## Pipeline Step 2: Create Sorting Bucket

Create a hollow bucket with a funnel shape for sorting LEGO parts.

**Script**: `blender/create_sorting_bucket.py`  
**Features**:
- Frustum shape (24cm top → 6cm bottom)
- 1cm thick walls
- Boolean operations for hollow geometry
- Added to 'bucket' collection

In [17]:
# Create the sorting bucket
success = client.execute_script_file(
    'blender/create_sorting_bucket.py',
    'Create Sorting Bucket',
    timeout=120
)

if success:
    print("\n✅ Sorting bucket created successfully")
else:
    print("\n❌ Failed to create sorting bucket")

📝 Executing script: blender/create_sorting_bucket.py
✅ Successfully executed Create Sorting Bucket!
Result:
✅ Created hollow bucket with funnel shape using boolean operations
✅ Created sorting bucket: Sorting_Bucket


✅ Sorting bucket created successfully


## Pipeline Step 3: Create Conveyor Belt

Create an inclined conveyor belt system to transport LEGO parts.

**Script**: `blender/create_conveyor_belt.py`  
**Features**:
- 1.5m length
- ~8.6° incline
- High friction coefficient (0.8)
- Added to 'conveyor_belt' collection

In [18]:
# Create the conveyor belt
success = client.execute_script_file(
    'blender/create_conveyor_belt.py',
    'Create Conveyor Belt',
    timeout=120
)

if success:
    print("\n✅ Conveyor belt created successfully")
else:
    print("\n❌ Failed to create conveyor belt")

📝 Executing script: blender/create_conveyor_belt.py
✅ Successfully executed Create Conveyor Belt!
Result:
🏗️ Creating conveyor belt system...
✓ Cleared existing conveyor belt objects
✓ Created conveyor belt collection
✓ Created conveyor belt base mesh
✓ Added conveyor belt details
✓ Setup conveyor belt passive rigid body for part transport
✓ Setup conveyor belt material animation (visual)
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
Info: Nothing to bake
✓ Created path-driven animated slats for friction transport
✓ Created hole in bucket side wall
✓ Removed decorative Conveyor_Belt object, preserving path and slats
🎉 Co

## Pipeline Step 4: Import LEGO Parts

Import 70+ common LEGO parts from the LDraw library.

**Script**: `blender/import_lego_parts.py`  
**Requirements**: LDraw library installed at `/Applications/Studio 2.0/ldraw/parts/`  
**Features**:
- Imports from most common LEGO parts list
- Arranges parts vertically with spacing
- Added to 'lego_parts' collection

⚠️ **Note**: This step takes ~5-10 minutes due to importing many parts

In [19]:
# Import LEGO parts (this takes several minutes)
print("⏳ Starting LEGO parts import (this may take 5-10 minutes)...")
print("   Importing 70+ parts from LDraw library")

success = client.execute_script_file(
    'blender/import_lego_parts.py',
    'Import LEGO Parts',
    timeout=600  # 10 minutes
)

if success:
    print("\n✅ LEGO parts imported successfully")
else:
    print("\n❌ Failed to import LEGO parts")

⏳ Starting LEGO parts import (this may take 5-10 minutes)...
   Importing 70+ parts from LDraw library
📝 Executing script: blender/import_lego_parts.py
✅ Successfully executed Import LEGO Parts!
Result:
🧱 Starting LEGO parts import...
📦 Found 10 LEGO parts to import
11:26:34.93 [importldraw] The LDraw Parts Library path to be used is: /Applications/Studio 2.0/ldraw
11:26:34.93 [importldraw] Use LSynth Parts requested
11:26:34.93 [importldraw] High-res primitives selected
11:26:34.94 [importldraw] Loading stud files
11:26:34.94 [importldraw] Loading files
11:26:34.97 [importldraw] Creating NodeGroups
11:26:34.97 [importldraw] Creating Blender objects
11:26:35.03 [importldraw] Number of vertices: 0
11:26:35.03 [importldraw] Number of convex hull vertices: 0
11:26:35.03 [importldraw] Adding 1 objects to scene
11:26:35.04 [importldraw] Load Done
Info: 11:26:34.93 [importldraw] The LDraw Parts Library path to be used is: /Applications/Studio 2.0/ldraw
Info: 11:26:34.93 [importldraw] Use LSy

## Pipeline Step 5: Setup Physics Animation

Configure rigid body physics for the LEGO parts simulation.

**Script**: `blender/animate_lego_physics.py`  
**Features**:
- Rigid body setup for all LEGO parts
- Mass: 2g per part
- Friction: 0.9 (high)
- Gravity: 9.81 m/s²
- 250 frame simulation

In [20]:
# Setup physics animation
success = client.execute_script_file(
    'blender/animate_lego_physics.py',
    'Setup Physics Animation',
    timeout=300
)

if success:
    print("\n✅ Physics animation configured successfully")
else:
    print("\n❌ Failed to setup physics animation")

📝 Executing script: blender/animate_lego_physics.py
✅ Successfully executed Setup Physics Animation!
Result:
🔬 Setting up LEGO physics simulation...
✅ Physics world configured with realistic gravity
ℹ️  Using simplified functional organization (bucket, conveyor_belt, lego_parts)
✅ Created physics ground plane
✅ Physics setup for bucket: Sorting_Bucket
📦 Found 10 LEGO parts for physics simulation
✅ Physics setup for LEGO part: 00000_3024.dat
✅ Physics setup for LEGO part: 00000_54200.dat
✅ Physics setup for LEGO part: 00000_3020.dat
✅ Physics setup for LEGO part: 00000_3022.dat
✅ Physics setup for LEGO part: 00000_3023.dat
✅ Physics setup for LEGO part: 00000_4073.dat
✅ Physics setup for LEGO part: 00000_3069b.dat
✅ Physics setup for LEGO part: 00000_2780.dat
✅ Physics setup for LEGO part: 00000_3710.dat
✅ Physics setup for LEGO part: 00000_3005.dat
✅ Assigned unique colors to 10 LEGO parts
✅ Added random variation to starting positions
ℹ️  Skipping bake (MCP mode); using manual per-fra

## Optional: Setup Lighting

Add three-point lighting to the scene for better visualization.

**Script**: `blender/setup_lighting.py`

In [21]:
# Setup lighting (optional)
success = client.execute_script_file(
    'blender/setup_lighting.py',
    'Setup Lighting',
    timeout=60
)

if success:
    print("\n✅ Lighting setup successfully")
else:
    print("\n❌ Failed to setup lighting")

📝 Executing script: blender/setup_lighting.py
✅ Successfully executed Setup Lighting!
Result:
💡 Setting up realistic room lighting…
✅ Lighting setup complete


✅ Lighting setup successfully


## Validation and Debugging

Run validation scripts to check scene state.

In [22]:
# Validate scene state
success = client.execute_script_file(
    'utils/validate_scene.py',
    'Validate Scene',
    timeout=60
)

if success:
    print("\n✅ Scene validation complete")
else:
    print("\n❌ Scene validation failed")

📝 Executing script: utils/validate_scene.py
✅ Successfully executed Validate Scene!
Result:

LEGO Sorter Scene Validation

Scene Statistics:
----------------------------------------
  collections         : 5
  objects             : 37
  meshes              : 122
  materials           : 13
  cameras             : 0
  lights              : 3
  rigidbodies         : 33
  frame_range         : 1-100

❌ Scene validation failed with 5 issue(s):

  1. Bucket cylinder not found in bucket collection
  2. Bucket base not found in bucket collection
  3. Conveyor_Belt object not found in scene
  4. SorterCam not found (run setup_lighting.py)
  5. No cameras in scene

💡 Tip: Run the full pipeline with 'python run_lego_sorter.py'




✅ Scene validation complete


## Custom Blender Code Execution

Execute custom Blender Python code directly from the notebook.

**Use case**: Quick experiments without creating new script files

In [23]:
# Example: Get scene statistics
custom_code = """
import bpy

# Count objects
total_objects = len(bpy.data.objects)
print(f"Total objects in scene: {total_objects}")

# List collections
print("\\nCollections:")
for col in bpy.data.collections:
    obj_count = len(col.objects)
    print(f"  - {col.name}: {obj_count} objects")

# Physics info
rigid_body_count = sum(1 for obj in bpy.data.objects if obj.rigid_body is not None)
print(f"\\nRigid body objects: {rigid_body_count}")
"""

success = client.execute_code(custom_code, 'Get Scene Statistics')

if not success:
    print("\n❌ Failed to execute custom code")

✅ Successfully executed Get Scene Statistics!
Result:
Total objects in scene: 37

Collections:
  - bucket: 1 objects
  - conveyor_belt: 21 objects
  - lego_parts: 10 objects
  - lighting: 3 objects
  - RigidBodyWorld: 33 objects

Rigid body objects: 33



## Summary

This notebook demonstrates:

✅ **Interactive pipeline control** - Run individual steps as needed  
✅ **Inline documentation** - Markdown cells explain each step  
✅ **Separate script files** - Blender scripts remain in `blender/` directory  
✅ **Custom code execution** - Quick experiments without file creation  
✅ **Error handling** - Clear success/failure feedback  

### Next Steps

- Add visualization cells (render images, show stats)
- Create parameter controls (sliders, dropdowns)
- Add progress bars for long-running operations
- Integrate pytest results

### Comparison with Python Scripts

**Notebook Advantages**:
- Interactive experimentation
- Inline documentation and visualizations
- Step-by-step execution

**Python Script Advantages**:
- Better version control (no JSON bloat)
- Easier testing with pytest
- Better IDE support (type hints, linting)
- Automation/CI-CD friendly